<!-- NINO26-CABECALHO v1 -->
# 8A — Distribuição no Brasil com redes neurais ConvLSTM (encoder-decoder)

**Projeto NINO-BRASIL — Oceanografia Física UFPE — Thiago Vilar**  
**Código da fase/letra:** `8A`  ·  **Hipótese:** HIP5 (ConvLSTM Brasil)

## Descritivo (por que este notebook existe)
Testa se uma rede espaço-temporal mapeia a sequência do Pacífico no campo futuro de chuva do Brasil com skill superior às Fases 4 e 6 — teleconexão como mapeamento espaço->espaço condicionado ao tempo.

## Pergunta
A rede aprende a distribuição espaço-temporal da chuva do Brasil com skill superior às Fases 4 e 6, por pixel/região/bioma?

## Desafio (hipótese a testar)
Alinhamento temporal correto (sem parear chuva anterior com Pacífico posterior), decoder de baixa dimensão (não Dense gigante), máscara/pesos de área e loss probabilística.

## Metodologia (com referências)
Encoder-decoder convolucional com alinhamento por timestamps, decoder EOF/convolucional de baixo posto, máscara Brasil e loss probabilística; métricas por pixel/região/bioma (Shi et al., 2015; Ham et al., 2019; Bachèlery et al., 2025; Chattopadhyay et al., 2020).

## Contrato de saídas — código predecessor único
Cada figura nasce do **mesmo** `registrar_figura(...)` que congela sua numeric-table sob o **mesmo código**, reescrevendo por **sobreposição** a cada execução:

```python
from nino_brasil.viz import registrar_figura
registrar_figura(fig, "Fig_8A01", fase=8, bloco="A",
                 titulo=..., descricao=..., hipotese="HIP5",
                 notebook="notebooks/fase8/8_brasil_convlstm.ipynb",
                 fontes={"<tabela>": df})   # -> figures/fase8/<codigo>.png + numeric-tables/fase8/<codigo>/
```

| Código | Figura (`figures/fase8/<código>.png`) | Numeric-table (`numeric-tables/fase8/<código>/`) | Descrição |
|---|---|---|---|
| `Fig_8A01` | `Fig_8A01.png` | `Fig_8A01/` | campo previsto de anomalia de chuva (a treinar) |
| `Fig_8A02` | `Fig_8A02.png` | `Fig_8A02/` | skill por pixel/região (a treinar) |

> Padrão em `docs/PADRAO_NOTEBOOKS.md`; validação por `python scripts/validar_figuras.py --strict`.

In [1]:
import sys, pathlib
ROOT=pathlib.Path.cwd()
while not (ROOT/'src').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src')); sys.path.insert(0,str(ROOT))
import pandas as pd; from IPython.display import Image, display
STATS=ROOT/'data/processed/parquet/statistics'


In [2]:
import nino_brasil.models.phase8_convlstm_brazil as p8
import numpy as np
pac=np.random.rand(80,24,8,24,3).astype('float32')  # (amostras, seq, lat, lon, canal)
br =np.random.rand(120,40,40).astype('float32')      # campos de chuva do Brasil
Xp,yb=p8.align_pacific_to_brazil(pac, br, horizon_weeks=4)
print('pares Pacifico->Brasil:', Xp.shape, yb.shape)


pares Pacifico->Brasil: (80, 24, 8, 24, 3) (80, 40, 40)


## Construcao do encoder-decoder (requer TensorFlow)
```python
model = p8.build_convlstm_encoder_decoder(pacific_shape=Xp.shape[1:], brazil_shape=(40,40))
model.fit(Xp, yb, epochs=30, validation_split=0.2, shuffle=False)
```
So avanca se vencer climatologia, persistencia, Fase 4 e Fase 6.


<!-- NINO26-REFERENCIAS v1 -->
## Referências Bibliográficas

1. Shi, X., et al. (2015). Convolutional LSTM Network. *NeurIPS 28*. https://arxiv.org/abs/1506.04214
2. Ham, Y.-G., Kim, J.-H., & Luo, J.-J. (2019). Deep learning for multi-year ENSO forecasts. *Nature*, 573, 568-572. https://doi.org/10.1038/s41586-019-1559-7
3. Bachèlery, M.-L., et al. (2025). Predicting Atlantic and Benguela Niño events with deep learning. *Science Advances*, 11, eads5185. https://doi.org/10.1126/sciadv.ads5185
4. Chattopadhyay, A., et al. (2020). Predicting clustered weather patterns (CNN espaço-temporal). *Scientific Reports*, 10, 1317. https://doi.org/10.1038/s41598-020-57897-9

Relação completa em `Artigos_Referências/Referências_Bibliográficas.xls`.